In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, load_npz
from sklearn.svm import LinearSVC
from tqdm import tqdm
import pickle
import gzip
import re
import os

In [2]:
def readGz(path):
    for l in gzip.open(path, 'rt'):
        yield eval(l)

def readCSV(path):
    f = gzip.open(path, 'rt')
    f.readline()
    for l in f:
        yield l.strip().split(',')

def clean(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9\s']", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

data = [d for d in readGz("train_Category.json.gz")]
texts = [clean(d["review_text"]) for d in data]
y = [d["genreID"] for d in data]

In [6]:
Xw_file = "Xw_features.npz"
Xc_file = "Xc_features.npz"
word_file = "vectorize_words.pkl"
char_file = "vectorize_chars.pkl"
    
def load_text_features(read=True):
    if read:
        print("Loading saved vectorized features...")
        Xw = load_npz(Xw_file)
        Xc = load_npz(Xc_file)
        with open(word_file, 'rb') as f:
            word_vec = pickle.load(f)
        with open(char_file, 'rb') as f:
            char_vec = pickle.load(f)
        print(f"Loaded Xw shape: {Xw.shape}, Xc shape: {Xc.shape}")
    else:
        print("Computing TF-IDF features (this may take a while)...")
        word_vec = TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,4),
            min_df=2,
            stop_words="english"
        )
        char_vec = TfidfVectorizer(
            analyzer="char",
            ngram_range=(3,5),
            min_df=2,
            max_features=30000
        )
        Xw = word_vec.fit_transform(texts)
        Xc = char_vec.fit_transform(texts)
        
    return Xw, Xc, word_vec, char_vec

In [7]:
Xw, Xc, word_vec, char_vec = load_text_features(read=False)

X = hstack([Xw, Xc])
n = X.shape[0]
cut = 9 * n // 10

Xtrain = X[:cut]
ytrain = y[:cut]
Xvalid = X[cut:]
yvalid = y[cut:]

Computing TF-IDF features (this may take a while)...


In [8]:
model_file = "baseline_model.pkl"
if os.path.exists(model_file):
    print("Loading saved baseline model...")
    with open(model_file, 'rb') as f:
        mod = pickle.load(f)
    print(f"Model loaded. Validation score: {mod.score(Xvalid, yvalid)}")
else:
    print("Training baseline model (this may take a while)...")
    mod = LinearSVC(C=0.1)
    mod.fit(Xtrain, ytrain)
    score = mod.score(Xvalid, yvalid)
    print(f"Model trained. Validation score: {score}")
    with open(model_file, 'wb') as f:
        pickle.dump(mod, f)
    print("Model saved.")

Training baseline model (this may take a while)...
Model trained. Validation score: 0.7709
Model saved.


In [ ]:
predictions = open("predictions_Category.csv", "w")
predictions.write("userID,reviewID,prediction\n")

for l in tqdm(readGz("test_Category.json.gz"), desc="Writing category predictions"):
    text = clean(l["review_text"])

    Xw = word_vec.transform([text])
    Xtest = Xw

    pred = mod.predict(Xtest)[0]
    predictions.write(f"{l['user_id']},{l['review_id']},{pred}\n")

predictions.close()

Writing category predictions: 0it [00:00, ?it/s]


ValueError: X has 50000 features, but LinearSVC is expecting 80000 features as input.